In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import glob
import os
from collections import Counter
from pypdf import PdfReader
import string

pdf_folder = "Document_Corpus"
corpus = []
metadata = []
remove_table = str.maketrans("", "", string.punctuation)

pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)

    try:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                corpus.append(text)
                metadata.append(
                    {
                        "filename": filename,
                        "page": page_num + 1,
                        "raw_text": text.strip(),
                    }
                )

    except Exception as e:
        print(f"Error reading{filename}:{e}")
print(f"Loaded {len(corpus)} pages from {len(pdf_files)} PDF Files")

Loaded 92 pages from 4 PDF Files


In [34]:
vectorizer = TfidfVectorizer()
Tf_Idfmatrix = vectorizer.fit_transform(corpus)

In [35]:
def search(query, top_n=5):
    query_vector = vectorizer.transform([query])
    similiarity_scores = cosine_similarity(query_vector, Tf_Idfmatrix).flatten()
    top_idx = np.argsort(similiarity_scores)[::-1][:top_n]

    match_found = False
    for rank, idx in enumerate(top_idx):
        score = similiarity_scores[idx]
        if score > 0:
            match_found = True

            if "metadata" in globals() and len(metadata) > idx:
                meta = metadata[idx]
                print(f"Rank {rank + 1} - Score {score:.2f}")
                print(f"File: {meta['filename']}(Page {meta['page']})")
                snippet = meta["raw_text"].replace("\n", "")[:200]
                print(f"Snippet: {snippet}...")

            else:
                print(f"rank {rank+1} - Score {score:.2f}")
                print(f"{corpus[idx]}")

    if not match_found:
        print("No matching document found with score > 0.")
    
    return top_idx

In [36]:
results = search("Random Forest")

Rank 1 - Score 0.11
File: Deep Learning for Anomaly Detection.pdf(Page 17)
Snippet: Deep Learning for Anomaly Detection: A Review 38:17Minimizing the loss in Equation (24) guarantees that the random nearest neighbor distance ofanomalies are at leastm greater than that of normal insta...
Rank 2 - Score 0.06
File: DeepAnT_A_Deep_Learning_Approach_for_Unsupervised_Anomaly_Detection_in_Time_Series.pdf(Page 4)
Snippet: M. Munir et al.: DeepAnT: Deep Learning Approach for Unsupervised Anomaly Detection in Time Seriesdata as input and learns the features individually. This isfollowed by a layer of MLP to perform class...
Rank 3 - Score 0.06
File: Deep Learning for Anomaly Detection.pdf(Page 16)
Snippet: 38:16 G. Pang et al.Due to some intrinsic differences between anomalies and normal instances presented in the self-supervisedclassifiers,thisapproachisalsoabletoworkinanunsupervisedsetting[ 157],demon...
Rank 4 - Score 0.04
File: GANomaly Semi-Supervised Anomaly.pdf(Page 10)
Snippet: 10 S. Akc